Imports

In [1]:
import os
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain.agents import Tool, initialize_agent
from langchain_neo4j import Neo4jGraph
from neo4j import GraphDatabase

load_dotenv()

True

Load variables


In [2]:
try:
    driver = GraphDatabase.driver(
        os.environ.get("NEO4J_URI"),
        auth=(os.environ.get("NEO4J_USERNAME"), os.environ.get("NEO4J_PASSWORD"))
    )

    driver.verify_connectivity()
    print("Neo4j connection successful")
    
    graph = Neo4jGraph(
        url=os.environ.get("NEO4J_URI"),
        username=os.environ.get("NEO4J_USERNAME"),
        password=os.environ.get("NEO4J_PASSWORD"),
        timeout=30  
    )
    

    llm = ChatAnthropic(
        model="claude-3-5-haiku-latest",
        temperature=0.3,
        anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
        max_tokens=4000
    )
    
    print("LLM initialized successfully")

except Exception as e:
    print(f"Setup failed: {e}")
    raise

Neo4j connection successful
LLM initialized successfully


Create Tools and Agent

In [ ]:
def create_road_network_tools():

    # def find_connected_links(link_id: str, levels: int = 2) -> str:
    #     with driver.session() as session:
    #         query = f"""
    #         MATCH (start:Link {{link_id: $link_id}})-[:CONNECTED_TO*1..{levels}]-(neighbor:Link)
    #         RETURN start.link_id as start_link, neighbor.link_id as connected_link, 
    #                neighbor.meters as distance
    #         LIMIT 50
    #         """
    #         result = session.run(query, link_id=link_id)
    #         return str([dict(record) for record in result])

    # def find_nearby_vms(link_id: str) -> str:
    #     with driver.session() as session:
    #         query = """
    #         MATCH path = (l:Link {link_id: $link_id})-[:CONNECTED_TO*0..2]-(connected:Link)<-[:LOCATED_AT]-(v:VMS)
    #         WITH v, connected, l, path, length(path) as hops,
    #             CASE 
    #             WHEN length(path) = 0 THEN 0
    #             ELSE reduce(total = 0, node IN nodes(path)[0..-1] | 
    #                         total + coalesce(node.meters, 0))
    #             END as cumulative_distance
            
    #         // Determine direction based on path relationships  
    #         WITH v, connected, l, cumulative_distance, hops,
    #             CASE 
    #             WHEN hops = 0 THEN 'same_link'
    #             WHEN hops = 1 THEN 'adjacent'
    #             WHEN hops = 2 THEN 'nearby'
    #             ELSE 'distant'
    #             END as proximity,
    #             relationships(path) as rels
            
    #         // Simple upstream/downstream logic
    #         WITH v, connected, l, cumulative_distance, hops, proximity,
    #             CASE 
    #             WHEN hops = 0 THEN 'same_link'
    #             WHEN hops > 0 AND size(rels) > 0 THEN
    #                 CASE 
    #                 WHEN startNode(rels[0]) = l THEN 'downstream'
    #                 WHEN endNode(rels[0]) = l THEN 'upstream'
    #                 ELSE 'cross_connection'
    #                 END
    #             ELSE 'unknown'
    #             END as direction
            
    #         RETURN v.EQT_NO as vms_id, 
    #             v.ROAD_NAME as road, 
    #             connected.link_id as vms_link,
    #             cumulative_distance as distance_meters,
    #             hops as links_away,
    #             direction,
    #             proximity
    #         ORDER BY cumulative_distance, hops
    #         LIMIT 20
    #         """
    #         result = session.run(query, link_id=link_id)
    #         return str([dict(record) for record in result])
        
    # def find_nearby_events(link_id: str) -> str:
    #     with driver.session() as session:
    #         query = """
    #         MATCH (l:Link {link_id: $link_id})-[:CONNECTED_TO*0..2]-(connected:Link)
    #         MATCH (connected)<-[:START_AT|END_AT]-(e:event_record)
    #         RETURN e.id as event_id, e.ROAD_NAME as road, connected.link_id as link
    #         LIMIT 20
    #         """
    #         result = session.run(query, link_id=link_id)
    #         return str([dict(record) for record in result])
        
    # def find_connected_plan(event_id: str) -> str:
    #     with driver.session() as session:
    #         query = """
    #         MATCH (e:event_record {id: $event_id})-[:HAS_PLAN]->(ep:event_plan)
    #         RETURN e.id as event_id,
    #             e.road_name as event_road,
    #             ep.id as plan_id,
    #             ep.plan_status as plan_status,
    #             ep.created_date as plan_created_date,
    #             ep.start_time as plan_start_time,
    #             ep.end_time as plan_end_time
    #         ORDER BY ep.created_date DESC
    #         LIMIT 20
    #         """
    #         result = session.run(query, event_id=int(event_id))
    #         return str([dict(record) for record in result])
    
    # def find_connected_plan_command(plan_id: str) -> str:
    #     with driver.session() as session:
    #         query = """
    #         MATCH (ep:event_plan {id: $plan_id})-[:HAS_COMMAND]->(epc:event_plan_command)
    #         RETURN ep.id as plan_id,
    #             ep.plan_status as plan_status,
    #             epc.id as command_id,
    #             epc.msgDesc1 as command_description_1,
    #             epc.msgDesc2 as command_description_2,
    #             epc.cmd_status as command_status,
    #             epc.created_date as command_created_date,
    #             epc.eqt_site_id as equipment_id
    #         ORDER BY epc.created_date DESC
    #         LIMIT 50
    #         """
    #         result = session.run(query, plan_id=int(plan_id))
    #         return str([dict(record) for record in result])


    def run_cypher_query(query: str) -> str:
        try:
            with driver.session() as session:
                result = session.run(query)
                records = [dict(record) for record in result]
                return str(records[:50])
        except Exception as e:
            return f"Query error: {str(e)}"



    def generate_smart_response_plan(context: str) -> str:
        
        prompt = f"""
        Based on the following traffic incident information, generate a comprehensive response plan:
        
        Context: {context}
        
        RESPONSE PLAN FORMAT:
        Please generate a response plan in the following JSON format:

        {{
            "plan_type": "Traffic Management Plan",
            "priority": "High/Medium/Low",
            "estimated_duration": "X minutes/hours",
            "traffic_management_actions": [
                            {{
                    "eqt_no": "VMS_EQUIPMENT_ID" (GET THIS FROM EQT_NO or EQT_EXT_ID),
                    "message_line_1": "Primary message text",
                    "message_line_2": "Secondary message text (if needed)",
                    "display_duration": "X minutes",
                    "justification": "Why this VMS and message"
                }},

                {{
                    "action": "Specific action to take",
                    "location": "Where to implement",
                    "resources_needed": "What resources are required",
                    "timing": "When to implement"
                }}
            ],
            
            "justification": "Detailed explanation of why this plan is appropriate"
        }}

        Generate a practical, actionable response plan considering the event severity, available VMS systems, and traffic management best practices.
        """
        
        try:
            response = llm.invoke(prompt)
            return response.content
        except Exception as e:
            return f"Error generating smart response plan: {str(e)}"
    
    # def find_affected_area_comprehensive(link_ids_input):
    #     """
    #     Find affected links using multiple pathfinding strategies.
    #     Input: link_ids_input (string or list) - Can be "[17840003591931, 17840001783588]" or actual list
    #     Returns: List of affected link IDs using fallback methods if direct path not found.
    #     """
        
    #     # Handle different input formats
    #     if isinstance(link_ids_input, str):
    #         # Try to parse string representation of list
    #         import ast
    #         import json
    #         try:
    #             # Try literal_eval first
    #             link_ids = ast.literal_eval(link_ids_input)
    #         except:
    #             try:
    #                 # Try JSON parsing
    #                 link_ids = json.loads(link_ids_input)
    #             except:
    #                 try:
    #                     # Try comma-separated values
    #                     link_ids = [x.strip() for x in link_ids_input.split(',')]
    #                 except:
    #                     return "Error: Could not parse link_ids input format"
    #     else:
    #         link_ids = link_ids_input
        
    #     # Validate we have exactly 2 link IDs
    #     if not isinstance(link_ids, list) or len(link_ids) < 2:
    #         return "Error: Need exactly 2 link IDs - upstream and downstream"
        
    #     up_link_id = str(link_ids[0])
    #     dn_link_id = str(link_ids[1])
        
    #     with driver.session() as session:
    #         query = """
    #         MATCH (start_link:Link {link_id: $up_link_id})
    #         MATCH (end_link:Link {link_id: $dn_link_id})
            
    #         // Method 1: Direct shortest path
    #         OPTIONAL MATCH path1 = shortestPath((start_link)-[:CONNECTED_TO*]-(end_link))
    #         WITH start_link, end_link, 
    #             CASE WHEN path1 IS NOT NULL 
    #                 THEN [node IN nodes(path1) | node.link_id] 
    #                 ELSE [] END as direct_path
            
    #         // Method 2: If no direct path, find links within reasonable distance of both start and end
    #         OPTIONAL MATCH (start_link)-[:CONNECTED_TO*1..3]-(intermediate:Link)-[:CONNECTED_TO*1..3]-(end_link)
    #         WITH start_link, end_link, direct_path,
    #             CASE WHEN size(direct_path) > 0 
    #                 THEN direct_path 
    #                 ELSE collect(DISTINCT intermediate.link_id)[0..10] END as affected_links
            
    #         // Method 3: If still no path, use links near start and end
    #         WITH start_link, end_link, 
    #             CASE WHEN size(affected_links) > 0 
    #                 THEN affected_links 
    #                 ELSE [start_link.link_id, end_link.link_id] END as final_affected_links
            
    #         RETURN $up_link_id as up_link_id,
    #             $dn_link_id as dn_link_id,
    #             final_affected_links as affected_links
    #         """
            
    #         result = session.run(query, up_link_id=up_link_id, dn_link_id=dn_link_id)
    #         return str([dict(record) for record in result])
        
    # def intelligent_vms_selection(event_context_input: str) -> str:

    #     import json
    #     import ast
        
    #     try:
    #         # Parse the input context
    #         if isinstance(event_context_input, str):
    #             try:
    #                 event_context = json.loads(event_context_input)
    #             except:
    #                 try:
    #                     event_context = ast.literal_eval(event_context_input)
    #                 except:
    #                     return "Error: Could not parse event context. Please provide valid JSON format."
    #         else:
    #             event_context = event_context_input
            
    #         # Extract key parameters
    #         affected_links = event_context.get('affected_links', [])
    #         event_severity = event_context.get('event_severity', 'medium')
    #         queue_length = event_context.get('queue_length', 'unknown')
    #         lane_blockage = event_context.get('lane_blockage', 'unknown')
    #         event_type = event_context.get('event_type', 'accident')
    #         time_of_day = event_context.get('time_of_day', '1200')
    #         weather_conditions = event_context.get('weather_conditions', 'clear')
    #         expected_duration = event_context.get('expected_duration', '30')
    #         road_type = event_context.get('road_type', 'highway')
    #         traffic_volume = event_context.get('traffic_volume', 'medium')
    #         incident_description = event_context.get('incident_description', '')
            
    #         if not affected_links:
    #             return "Error: No affected links provided in context"
            
    #         # Get comprehensive VMS data for the affected area
    #         with driver.session() as session:
    #             # Advanced VMS query that considers multiple factors
    #             vms_query = """
    #             UNWIND $link_ids as link_id
    #             MATCH path = (incident_link:Link {link_id: link_id})-[:CONNECTED_TO*0..5]-(connected_link:Link)<-[:LOCATED_AT]-(vms:VMS)
                
    #             WITH vms, connected_link, incident_link, path, length(path) as hop_distance, link_id,
    #                 // Calculate physical distance
    #                 CASE 
    #                 WHEN length(path) = 0 THEN 0
    #                 ELSE reduce(total = 0, node IN nodes(path)[0..-1] | 
    #                             total + coalesce(node.meters, 0))
    #                 END as physical_distance_meters,
                    
    #                 // Determine traffic flow direction
    #                 relationships(path) as path_rels
                
    #             WITH vms, connected_link, incident_link, hop_distance, link_id, physical_distance_meters,
    #                 CASE 
    #                 WHEN hop_distance = 0 THEN 'incident_location'
    #                 WHEN hop_distance > 0 AND size(path_rels) > 0 THEN
    #                     CASE 
    #                     WHEN startNode(path_rels[0]) = incident_link THEN 'downstream'
    #                     WHEN endNode(path_rels[0]) = incident_link THEN 'upstream'
    #                     ELSE 'cross_traffic'
    #                     END
    #                 ELSE 'unknown_direction'
    #                 END as traffic_direction,
                    
    #                 // Calculate strategic importance score
    #                 CASE 
    #                 WHEN hop_distance = 0 THEN 10  // At incident location
    #                 WHEN hop_distance = 1 AND startNode(path_rels[0]) = incident_link THEN 9  // Immediate downstream
    #                 WHEN hop_distance = 1 AND endNode(path_rels[0]) = incident_link THEN 8   // Immediate upstream
    #                 WHEN hop_distance = 2 AND startNode(path_rels[0]) = incident_link THEN 7  // 2 hops downstream
    #                 WHEN hop_distance = 2 AND endNode(path_rels[0]) = incident_link THEN 6   // 2 hops upstream
    #                 WHEN hop_distance <= 3 THEN 5  // Within 3 hops
    #                 WHEN hop_distance <= 5 THEN 3  // Within 5 hops
    #                 ELSE 1
    #                 END as strategic_score
                
    #             // Get additional VMS and road context
    #             OPTIONAL MATCH (connected_link)-[:START_AT|END_AT]-(historical_events:event_record)
    #             WITH vms, connected_link, incident_link, hop_distance, physical_distance_meters, 
    #                  traffic_direction, strategic_score, link_id,
    #                  count(DISTINCT historical_events) as historical_incident_count
                
    #             // Get historical VMS usage patterns
    #             OPTIONAL MATCH (vms)<-[:CONTROLS_VMS]-(historical_commands:event_plan_command)
    #             WITH vms, connected_link, incident_link, hop_distance, physical_distance_meters,
    #                  traffic_direction, strategic_score, link_id, historical_incident_count,
    #                  collect(DISTINCT {
    #                      message1: historical_commands.msgDesc1,
    #                      message2: historical_commands.msgDesc2,
    #                      command_status: historical_commands.cmd_status
    #                  }) as historical_messages
                
    #             RETURN DISTINCT 
    #                 vms.ID as vms_id,
    #                 vms.EQT_NO as vms_no,
    #                 vms.ROAD_NAME as road_name,
    #                 vms.ROAD_CAT as road_category,
    #                 vms.LONGITUDE as longitude,
    #                 vms.LATITUDE as latitude,
    #                 vms.DIR as direction,
    #                 connected_link.link_id as vms_link_id,
    #                 link_id as incident_link_id,
    #                 hop_distance,
    #                 physical_distance_meters,
    #                 traffic_direction,
    #                 strategic_score,
    #                 historical_incident_count,
    #                 historical_messages[0..3] as recent_messages,  // Last 3 historical messages
                    
    #                 // Additional contextual scoring
    #                 CASE 
    #                 WHEN vms.ROAD_CAT = 'HIGHWAY' THEN 3
    #                 WHEN vms.ROAD_CAT = 'ARTERIAL' THEN 2
    #                 ELSE 1
    #                 END as road_importance_score,
                    
    #                 // Calculate composite effectiveness score
    #                 strategic_score + 
    #                 CASE WHEN vms.ROAD_CAT = 'HIGHWAY' THEN 3 ELSE 1 END +
    #                 CASE WHEN historical_incident_count > 5 THEN 2 ELSE 0 END as composite_score
                
    #             ORDER BY composite_score DESC, physical_distance_meters ASC
    #             LIMIT 30
    #             """
                
    #             result = session.run(vms_query, link_ids=affected_links)
    #             vms_candidates = [dict(record) for record in result]
            
    #         # Now let the LLM make intelligent decisions
    #         llm_selection_prompt = f"""
    #         You are a traffic management AI. Select which VMS signs to activate for this incident.

    #         INCIDENT CONTEXT:
    #         - Event Type: {event_type}
    #         - Severity Level: {event_severity}
    #         - Queue Length: {queue_length}
    #         - Lanes Blocked: {lane_blockage}
    #         - Time: {time_of_day}
    #         - Weather: {weather_conditions}
    #         - Expected Duration: {expected_duration} minutes
    #         - Road Type: {road_type}
    #         - Traffic Volume: {traffic_volume}
    #         - Description: {incident_description}
    #         - Affected Links: {affected_links}

    #         AVAILABLE VMS CANDIDATES:
    #         {json.dumps(vms_candidates, indent=2)}

    #         SELECTION CRITERIA:
    #         1. Prioritize upstream VMS signs to warn approaching traffic
    #         2. Consider distance from incident (optimal warning distances)
    #         3. Account for severity level and expected duration
    #         4. Select multiple VMS for comprehensive coverage
    #         5. Consider road hierarchy and traffic volume

    #         RESPONSE FORMAT:
    #         {{
    #             "selected_vms": [
    #                 {{
    #                     "vms_id": "equipment_id",
    #                     "priority": "Critical/High/Medium/Low",
    #                     "reasoning": "Brief explanation for selection",
    #                     "distance_from_incident": "X meters",
    #                     "traffic_direction": "upstream/downstream/cross"
    #                 }}
    #             ],
    #             "selection_summary": "Overall strategy and reasoning for VMS selection"
    #         }}

    #         Select the optimal VMS signs based on the incident context. Focus on which signs to use, not what messages to display.
    #         """
            
    #         # Get LLM decision
    #         response = llm.invoke(llm_selection_prompt)
    #         return response.content
            
    #     except Exception as e:
    #         return f"Error in VMS selection: {str(e)}"
     
    def get_schema(input_text: str = "") -> str:
        """Get comprehensive Neo4j database schema using APOC procedures.
        
        Args:
            input_text: Not used, but required for agent compatibility
            
        Returns:
            Complete database schema information including nodes, relationships, and properties
        """
        try:
            with driver.session() as session:
                # Get node labels and their properties
                nodes_query = """
                CALL apoc.meta.nodeTypeProperties()
                YIELD nodeType, nodeLabels, propertyName, propertyTypes, mandatory
                RETURN nodeType, nodeLabels, propertyName, propertyTypes, mandatory
                ORDER BY nodeType, propertyName
                """
                
                # Get relationship types and their properties
                rels_query = """
                CALL apoc.meta.relTypeProperties()
                YIELD relType, propertyName, propertyTypes, mandatory
                RETURN relType, propertyName, propertyTypes, mandatory
                ORDER BY relType, propertyName
                """
                
                # Get schema overview
                overview_query = """
                CALL apoc.meta.schema()
                YIELD value
                RETURN value
                """
                
                # Execute queries
                nodes_result = session.run(nodes_query)
                rels_result = session.run(rels_query)
                overview_result = session.run(overview_query)
                
                # Format results
                schema_info = {
                    "nodes": [dict(record) for record in nodes_result],
                    "relationships": [dict(record) for record in rels_result],
                    "overview": [dict(record) for record in overview_result]
                }
                
                return str(schema_info)
                
        except Exception as e:
            # Fallback to basic schema if APOC is not available
            try:
                with driver.session() as session:
                    # Basic schema without APOC
                    basic_query = """
                    CALL db.labels() YIELD label
                    WITH collect(label) as labels
                    CALL db.relationshipTypes() YIELD relationshipType
                    WITH labels, collect(relationshipType) as relationships
                    RETURN {
                        labels: labels,
                        relationships: relationships
                    } as schema
                    """
                    result = session.run(basic_query)
                    return str([dict(record) for record in result])
            except Exception as basic_error:
                return f"Schema error: {str(e)}. Basic fallback error: {str(basic_error)}"   
    
        
    return [
        # Tool(
        #     name="FindConnectedLinks",
        #     func=find_connected_links,
        #     description="Use this to find road links connected to a given link within specified levels"
        # ),
        # Tool(
        #     name="FindNearbyVMS", 
        #     func=find_nearby_vms,
        #     description="""Find VMS (Variable Message Signs) near a specific road link.
        #     Input: link_id (string) - The road link identifier to search around.
        #     Returns: List of nearby VMS with equipment IDs, distances, and directions."""
        # ),
        Tool(
            name="CypherQuery",
            func=run_cypher_query,
            description="Execute Cypher queries on the Neo4j graph database. Use this for general graph queries."
        ),
        # Tool(
        #     name="FindEventRecord",
        #     func=find_nearby_events,
        #     description="""Find Event Records near a specific road link.
        #     Input: link_id (string) - The road link identifier to search around.
        #     Returns: List of events with event IDs, roads, and associated links."""
        # ),
        # Tool(
        #     name="FindEventPlan",
        #     func=find_connected_plan,
        #     description="""Find the Event Plans connected to the event record
        #     Input: event_id (string) - The id of the event_record.
        #     Returns: list of Event plans with their details that were used for the event, 
        #     """
        # ),
        # Tool(
        #     name="FindPlanCommands",
        #     func=find_connected_plan_command,
        #     description="""Find the Event Plan Commands used by a specific event plan
        #     Input: plan_id (string) the id of the event plan.
        #     Returns: The information of the Event plan command"""
        # ),
        Tool(
            name="GenerateSmartResponsePlan",
            func=generate_smart_response_plan,
            description="""Generate intelligent response plan using incident context.
            Input: context (string) - Comprehensive context including incident details, VMS data importantly the id, event with their respective plan and plan command history to provide context.
            Returns: the explicit JSON formatted response plan with actions and VMS commands not the summary."""
        ),
        Tool(
            name = "GetSchema",
            func = get_schema,
            description = """Use this tool to get the Neo4j database schema (nodes and relationships). 
            COMPULSORY input str ""
            Output is a str of the graph schema
            """
        ),
        # Tool(
        #     name="FindAffectedAreaComprehensive",
        #     func=find_affected_area_comprehensive,
        #     description="""Find affected links using multiple pathfinding strategies.
        #     Input: link_ids (string) - Two link IDs as "[upstream_link_id, downstream_link_id]" or "upstream_link_id, downstream_link_id"
        #     Returns: List of affected link IDs using fallback methods if direct path not found."""
        # ),
        # Tool(
        #     name="IntelligentVMSSelection",
        #     func=intelligent_vms_selection,
        #     description="""Advanced VMS selection tool with full LLM autonomy for traffic incident management.
            
        #     Input: JSON string with event context including:
        #     - affected_links: List of affected link IDs
        #     - event_severity: low/medium/high/critical
        #     - queue_length: Queue length in meters or description
        #     - lane_blockage: Number of lanes blocked
        #     - event_type: accident/breakdown/construction/weather
        #     - time_of_day: HHMM format
        #     - weather_conditions: clear/rain/fog/etc
        #     - expected_duration: Duration in minutes
        #     - road_type: highway/arterial/local
        #     - traffic_volume: low/medium/high
        #     - incident_description: Detailed description
            
        #     Returns: JSON response with selected VMS signs and basic reasoning.
        #     Focuses on WHICH VMS signs to activate, not what messages to display.
        #     Multiple VMS signs can be selected for comprehensive coverage."""
        # )
    ]




Chat history query

In [49]:
def custom_parsing_error_handler(error):
    """Custom handler to see what the LLM actually outputs"""
    print("=" * 50)
    print("PARSING ERROR DETECTED!")
    print("Error message:", str(error))
    print("=" * 50)
    
    # Try to extract the actual LLM output from the error
    error_str = str(error)
    if "Could not parse LLM output:" in error_str:
        # Extract the actual output
        start_idx = error_str.find("Could not parse LLM output:") + len("Could not parse LLM output:")
        end_idx = error_str.find("For troubleshooting")
        if end_idx == -1:
            actual_output = error_str[start_idx:].strip()
        else:
            actual_output = error_str[start_idx:end_idx].strip()
        
        print("ACTUAL LLM OUTPUT:")
        print(actual_output)
        print("=" * 50)
        
        # Return a formatted response
        return f"LLM tried to use tool but format was wrong. Raw output: {actual_output}"
    
    return f"Parsing error: {error}"

class RoadNetworkChatBot:
    def __init__(self):
        self.chat_history = []
        self.road_tools = create_road_network_tools()
        
        schema_docs = """
        Link nodes represent segments of the road 
        The Link node have the following properties
        from_junction, string, the junction where the link starts from
        link_id, string, the unique id for the link
        meters, string, the length of the link in meters
        to_junction, string, the junction where the link ends at

        VMS nodes represent electronic warning signs on the road
        The VMS nodes have the following properties
        DIR, float, direction where it is facing 1 for forward 2 for backwards
        DIST_TO_UPNODE, integer, distance in meters to the up node
        EQT_EXT_ID, string, the equipment unique id number
        EQT_NO, string, the equipment unique id number
        EQT_TYPE, string, the type of equipment
        ID, integer, 3-digit id of equipment
        LATITUDE, float, latitude position of equipment
        LINK_ID, integer, unique id of link where equipment is positioned
        LONGITUDE, float, longitude position of equipment
        ROAD_CAT, string, category of road equipment is placed at
        ROAD_CODE, string, code of road equipment is placed at
        ROAD_NAME, string, name of road equipment is placed at

        RELATIONSHIPS:
        The Link node and VMS node are connected by relationship LOCATED_AT which indicates where the VMS is located at it is always VMS to Link.
        They are connected by matching identical VMS (LINK_ID) to Link (link_id),
        when matching ensure for VMS it is an integer and for Link it is a string

        The Link nodes are connected by relationship CONNECTED_TO which represents the physical road network topology.
        CONNECTED_TO relationships form bidirectional connections between adjacent road segments.
        Links are connected by matching Link (to_junction) with Link (from_junction) of adjacent segments.

        """

        system_prompt = f"""
       You are a traffic incident management expert.

        CRITICAL REQUIREMENTS - DO NOT DEVIATE:
        1. ALWAYS start by performing GetSchema tool INPUT MUST BE "" WHEN DOING SO
        2. For incident response, you MUST find VMS signs UPSTREAM of the incident
        3. MANDATORY: Search exactly 50 road links upstream using CONNECTED_TO*1..50
        4. FORBIDDEN: Omnidirectional search, downstream search, or limited hop search
        5. REQUIRED: Use this exact pattern: (incident)<-[:CONNECTED_TO*1..50]-(upstream)<-[:LOCATED_AT]-(vms)
        6. CONTINUE: If no VMS found within 50 keep going another 20 additional hops

        CORRECT CYPHER PATTERN EXAMPLES:
        ✅ CORRECT: MATCH (incident:Link) WHERE incident.link_id = '17840002118812'
        MATCH (incident)<-[:CONNECTED_TO*1..50]-(upstream:Link)
        MATCH (upstream)<-[:LOCATED_AT]-(vms:VMS)

        ❌ WRONG: MATCH (upstream:Link)<-[:CONNECTED_TO*1..50]-(incident:Link)
        ❌ WRONG: MATCH (incident:Link)-[:CONNECTED_TO*1..50]->(upstream:Link)

        DIRECTION RULE: Arrow MUST point toward the incident: upstream -> incident
        This means: (incident)<-[:CONNECTED_TO*1..50]-(upstream)

        {schema_docs}
        Use the documentation above to understand what each node/edge means

        TRAFFIC FLOW RULES:
        - Upstream = where traffic comes FROM (toward incident) 
        - Use arrow syntax: incident<-[:CONNECTED_TO*1..50]-upstream
        - This finds links where traffic flows toward the incident

        Your mission: Find upstream VMS to warn approaching drivers and prevent secondary accidents.
        """

        self.agent = initialize_agent(
            tools=self.road_tools,
            llm=llm,
            agent="chat-conversational-react-description",
            verbose=True,
            max_iterations=8,
            return_intermediate_steps=True,
            handle_parsing_errors=custom_parsing_error_handler,
            agent_kwargs={
                "system_message": system_prompt
            }
        )
    
    def query(self, question: str):
        try:
            response = self.agent.invoke({
                "input": question,
                "chat_history": self.chat_history
            })
            
            self.chat_history.extend([
                ("human", question),
                ("ai", response.get("output", ""))
            ])
            
            return response.get("output", str(response))
        except Exception as e:
            return f"Error: {e}"
    
    def clear_history(self):
        self.chat_history = []
    
    def get_history(self):
        return self.chat_history

chatbot = RoadNetworkChatBot()

Test run

In [51]:
chatbot.clear_history()
print("History cleared",chatbot.get_history())

response1 = chatbot.query(
    "A car accident has occurred at link_id 17840005686171 , the time is 0800, the event severity is medium, the event type is accident"
)
print("Context gathered:", response1 )

# # Step 2: Generate response plan
# response2 = enhanced_session_chatbot.query("Generate an intelligent response plan for this situation")
# print("Response Plan:", response2)
# response = enhanced_session_chatbot.query("At this link there has been a car accident, severity is medium. Generate a response plan")
# print("Response Plan:", response)



History cleared []


> Entering new AgentExecutor chain...
```json
{
    "action": "CypherQuery",
    "action_input": "MATCH (incident:Link {link_id: '17840005686171'}) RETURN incident"
}
```
Observation: [{'incident': <Node element_id='4:20633528-6544-4636-aa46-fde868c3105c:111' labels=frozenset({'Link'}) properties={'from_junction': '17840007417598', 'to_junction': '17840007422237', 'link_id': '17840005686171', 'meters': '327.72'}>}]
Thought:```json
{
    "action": "CypherQuery",
    "action_input": "MATCH (incident:Link {link_id: '17840005686171'})<-[:CONNECTED_TO*1..50]-(upstream:Link) MATCH (upstream)<-[:LOCATED_AT]-(vms:VMS) RETURN DISTINCT vms"
}
```
Observation: []
Thought:```json
{
    "action": "CypherQuery",
    "action_input": "MATCH (incident:Link {link_id: '17840005686171'})<-[:CONNECTED_TO*1..70]-(upstream:Link) MATCH (upstream)<-[:LOCATED_AT]-(vms:VMS) RETURN DISTINCT vms"
}
```
Observation: []
Thought:```json
{
    "action": "GenerateSmartResponsePlan",
    "action_inp

### hide

In [7]:
# # response1 = session_chatbot.query( "First use FindAffectedAreaComprehensive with [17840006094894, 17840006090439], "
# #     "then use FindVMSForAffectedArea to find ALL VMS signs covering the affected area")

# # # Alternative approach - step by step
# # session_chatbot.clear_history()
# # print("History cleared")

# # print("=== Step 1: Find Affected Area ===")
# # response1 = session_chatbot.query(
# #     "Use FindAffectedAreaComprehensive with upstream link 17840003591931 and downstream link 17840001783588"
# # )
# # print("Affected area:", response1)

# # print("\n=== Step 2: Find VMS Signs ===")
# # response2 = session_chatbot.query(
# #     "Find VMS signs near link 17840003591931 and link 17840001783588"
# # )
# # print("VMS signs:", response2)

# # print("\n=== Step 3: Find Event Records ===")
# # response3 = session_chatbot.query(
# #     "Find event records near link 17840003591931 and link 17840001783588"
# # )
# # print("Event records:", response3)

# # print("\n=== Step 4: Generate Response Plan ===")
# # response4 = session_chatbot.query(
# #     "Based on all the previous information about the affected area, VMS signs, and event records, "
# #     "generate a comprehensive response plan for a medium severity accident at 22:30 between these links. "
# #     "The main objectives are to reduce secondary accidents and alleviate congestion."
# # )
# # print("Response plan:", response4)
# event_context = {
#     "affected_links": ['17840001659146', '17840001802107', '17840006026933', '17840006026934', '17840006121538', '17840006121200', '17840005723454', '17840005731114', '17840003596090', '17840002443857', '17840002000313', '17840004458759', '17840003979514', '17840004680271', '17840003596151', '17840003129228', '17840001921326', '17840002864884', '17840005610038', '17840005608110', '17840005609237', '17840001814427', '17840006013498', '17840006013499', '17840003225827', '17840004102372', '17840002674594', '17840002674595', '17840001965087', '17840006091898', '17840006111153', '17840006418106', '17840006418104', '17840002175464', '17840002175458', '17840002354082', '17840006417241', '17840006415565', '17840006418177', '17840006117199', '17840006411534', '17840006411162', '17840002250648', '17840004469360', '17840001256445', '17840003592357', '17840001918700', '17840003591931'],
#     "event_severity": "high",
#     "queue_length": "2000 meters",
#     "lane_blockage": "2 of 3 lanes",
#     "event_type": "accident",
#     "time_of_day": "0800",
#     "weather_conditions": "clear",
#     "expected_duration": "45",
#     "road_type": "highway",
#     "traffic_volume": "high",
#     "incident_description": "Multi-vehicle accident blocking 2 lanes during morning rush hour"
# }

# # Query the chatbot
# response = session_chatbot.query(
#     f"Use the IntelligentVMSSelection tool with this context: {json.dumps(event_context)} "
#     "to create the smart response plan system."
# )

In [8]:
# session_chatbot.set_link("17840003591931")

# response1 = session_chatbot.query("Find Event records Nearby that link?")
# print("Response 1:", response1)

# response2 = session_chatbot.query("What are the details of the event plan command connected to the events nearby?")
# print("Response 2:", response2)

# print("History:", session_chatbot.get_history())

do link to link connection ✅

complete rag prompt, need the query bot to understand the question to properly generate a cypher query to gather the right context, maybe could perform searches multiple times to get the best context, on top of this the understanding bot needs to create a usable response plane ie a json file, need it to convert the question using the context to create a accurate and usable response plan

response plan structure ✅

for event record the up link and dn link id might be refering to the id in the road map❌ This does not refer to anything and is just filler nonsense data

utilize q_end_link_id to find all the links that the event passes through, maybe can also utilize up & dn link to find the links inbetween 

add into link nodes the length of each link

edit the data for event record to have it be more than 1 link long, also edit the length of the event to make it work

explore the flow of the rag and how it finds similar events, do not use exact link_id
basically understand and develop similarity functions

give the llm spatial context, given the region where the event occurs it should know what vms it can use


## **Vector Index Search for RAG**

### **How it works:**
- Converts text/data into high-dimensional vectors (embeddings)
- Uses similarity search (cosine, euclidean) to find semantically similar content
- Typically uses vector databases like Pinecone, Weaviate, or Neo4j Vector Index

### **Pros:**
**Semantic Understanding**: Captures meaning, not just keywords  
**Fuzzy Matching**: Finds conceptually similar content even with different wording  
**Cross-language Support**: Works across languages with multilingual models  
**Handles Synonyms**: "car accident" matches "vehicle collision"  
**Fast Retrieval**: Optimized for similarity search at scale  

### **Cons:**
**No Relationship Context**: Doesn't understand connections between entities  
**Hallucination Risk**: May retrieve semantically similar but factually wrong content  
**Limited Precision**: Can miss exact matches for specific IDs/codes  
**Embedding Quality Dependent**: Results only as good as the embedding model  

---

## **Graph Cypher Search for RAG**

### **How it works:**
- Traverses graph relationships to find connected data
- Uses precise pattern matching and relationship queries
- Leverages graph structure for contextual retrieval

### **Pros:**
**Relationship Awareness**: Understands how entities connect  
**Precise Retrieval**: Exact matches for IDs, codes, structured data  
**Contextual Navigation**: Follows relationships for deeper context  
**No Hallucination**: Returns only existing, connected data  
**Complex Queries**: Multi-hop relationships, aggregations  

### **Cons:**
**Rigid Matching**: Requires exact schema knowledge  
**Limited Semantic Understanding**: Misses conceptual similarities  
**Query Complexity**: Complex traversals can be slow  
**Schema Dependent**: Breaks if relationships aren't modeled correctly  

---


## **Why Combining Vector + Graph Search is Beneficial**

### **Complementary Strengths**
- **Graph provides structure** - knows exactly which VMS signs are physically connected to which road links
- **Vector provides meaning** - understands that "traffic jam" and "congestion" refer to similar concepts
- **Together they bridge the gap** between spatial relationships and semantic understanding

### **Real-World Example from Your Domain**
Imagine a user asks: *"Find accident-related messages on signs near Sheikh Zayed Road"*

**Graph search alone** would find VMS signs spatially near Sheikh Zayed Road links, but might miss signs showing accident messages if they don't have exact keyword matches.

**Vector search alone** would find semantically similar content about accidents, but couldn't tell you which signs are actually geographically relevant to Sheikh Zayed Road.

**Combined approach** finds VMS signs that are:
1. **Spatially relevant** (near the road via graph relationships)
2. **Semantically relevant** (showing accident-related content via vector similarity)

### **Coverage Completeness**
- **Graph fills vector gaps** - ensures you don't miss nearby equipment just because descriptions don't match semantically
- **Vector fills graph gaps** - finds relevant content even when exact relationships aren't modeled
- **Redundancy as validation** - when both methods return the same result, you have higher confidence

### **Query Complexity Handling**
- **Simple spatial queries** → Graph handles efficiently
- **Simple semantic queries** → Vector handles efficiently  
- **Complex hybrid queries** → Both methods work together
- **Ambiguous queries** → LLM can try both approaches and compare results

---

## **Utilizing both methods together**

### **Dynamic Weight Assignment**
The LLM can analyze the query and automatically decide:

- **Spatial-heavy queries** (80% graph, 20% vector)
  - *"VMS signs on the next 3 links downstream"*
  - *"Events between junctions 5 and 7"*

- **Semantic-heavy queries** (20% graph, 80% vector)
  - *"Find traffic incidents similar to vehicle breakdowns"*
  - *"Show emergency-related messages"*

- **Balanced queries** (50% graph, 50% vector)
  - *"Accident messages on signs near this location"*
  - *"Traffic alerts in the vicinity"*

### **Adaptive Intelligence**
The LLM can:
- **Learn from context** - if it's a follow-up question about a specific link, prioritize graph search
- **Detect query intent** - recognize whether user wants spatial proximity or semantic similarity
- **Adjust based on results** - if graph search returns few results, increase vector weight
- **Consider data density** - in areas with sparse graph connections, rely more on vector search

### **Self-Improving System**
- **Result quality feedback** - LLM can observe which combination produces better answers
- **Query pattern recognition** - learn that certain question types work better with specific weightings
- **Context awareness** - understand that road network queries typically need spatial focus
- **Graceful degradation** - if one search method fails, automatically rely more on the other

This creates a self improving system where the LLM acts as a smart orchestrator, dynamically balancing both search methods based on query characteristics and context

Tomorrow i want to explore similarity searches

perhaps use a hybrid rag system involving vector search for sematic and cypher search for spatial

utilize feature search

for searching for nearby VMS have to implement a way to determin if VMS is in the same direction as the road where event occurs

also decide to what level the cypher search should go up to, maybe could use a distance method as each link has a distance

Compare both vector and Graph RAG using the same queries, road type, lane blockage, queue length to identify the similar events

For VMS similar search can be refined, is the knowledge graph required.